In [2]:
from pinecone import Pinecone, ServerlessSpec
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
openai_api_key = os.getenv("OPENAI_API_KEY")
model_name = os.getenv("OPENAI_EMBEDDING_MODEL_NAME")
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_index_name = os.getenv("PINECONE_INDEX_NAME")

### Initialize Pinecone


In [4]:
pinecone = Pinecone(
    api_key=pinecone_api_key
)


client = OpenAI(
    api_key=openai_api_key,
)

### Wrangle Dataset


In [10]:
dataframe = pd.read_json("./products/products.jsonl", lines=True)
dataframe.head()

,name,category,description,ingredients,price,rating,image_path
0,Cappuccino,Coffee,A rich and creamy cappuccino made with freshly...,"[Espresso, Steamed Milk, Milk Foam]",4.50,4.7,cappuccino.jpg
1,Jumbo Savory Scone,Bakery,"Deliciously flaky and buttery, this jumbo savo...","[Flour, Butter, Cheese, Herbs, Baking Powder, ...",3.25,4.3,SavoryScone.webp
2,Latte,Coffee,"Smooth and creamy, our latte combines rich esp...","[Espresso, Steamed Milk, Milk Foam]",4.75,4.8,Latte.jpg
3,Chocolate Chip Biscotti,Bakery,"Crunchy and delightful, this chocolate chip bi...","[Flour, Sugar, Chocolate Chips, Eggs, Almonds,...",2.50,4.6,chocolat_biscotti.jpg
4,Espresso shot,Coffee,"A bold shot of rich espresso, our espresso is ...",[Espresso],2.00,4.9,Espresso_shot.webp


In [15]:
dataframe["text"] = dataframe["name"] + " : " + dataframe["description"] + " -- Ingredients: " + dataframe["ingredients"].astype(str) + " -- Price: " + dataframe["price"].astype(str) + " -- Rating: " + dataframe["rating"].astype(str)

In [19]:
# convert the dataframe["text"] to list

text_list = dataframe["text"].tolist()


In [20]:
# adding about shop details to text_list list

with open("./products/Merry's_way_about_us.txt") as f:
    about_us = f.read()
    
coffee_shop_about = "Merry's Way Coffee Shop about section: " + about_us
text_list.append(coffee_shop_about)

In [22]:
# adding the menu list of the cofee shop

with open("./products/menu_items_text.txt") as f:
    menu_items = f.read()
    
menu_items = "Merry's Way Coffee Shop menu items: " + menu_items
text_list.append(menu_items)

### Generating the embeddings


In [23]:
embeddings = client.embeddings.create(
    input=text_list,
    model=model_name,
)

In [25]:
extract_embeddings = embeddings.data[0].embedding

len(extract_embeddings)

1536

### Adding data into the Vector DB


In [27]:
pinecone.create_index(
    name=pinecone_index_name,
    dimension=len(extract_embeddings),
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1",
    ),
)

{
    "name": "reception-bot",
    "metric": "cosine",
    "host": "reception-bot-cqzsuxv.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}

In [33]:
# wait for the index to be ready
import time

while not pinecone.describe_index(pinecone_index_name).status.ready:
    print("Waiting for index to be ready...")
    time.sleep(1)
    
# create the index
index = pinecone.Index(pinecone_index_name)

vectors = []

for text_li, emb in zip(text_list, embeddings.data):
    
    entry_id = text_li.split(":")[0]
    
    vectors.append({
        "id": entry_id,
        "values": emb.embedding,
        "metadata": {
            "text": text_li
        }
    })
    
# upsert the vectors to the index
index.upsert(
    vectors=vectors,
    namespace="ns1"
)


{'upserted_count': 20}

#### Get closest documents


In [44]:
output = client.embeddings.create(
    input="Is Cappuccino lactose-free?",
    model=model_name,
)

embedding = output.data[0].embedding

In [45]:
res = index.query(
    vector=embedding,
    top_k=5,
    namespace="ns1",
    include_values=False,
    include_metadata=True,
)

In [46]:
print(res)

{'matches': [{'id': 'Cappuccino ',
              'metadata': {'text': 'Cappuccino : A rich and creamy cappuccino '
                                   'made with freshly brewed espresso, steamed '
                                   'milk, and a frothy milk cap. This '
                                   'delightful drink offers a perfect balance '
                                   'of bold coffee flavor and smooth milk, '
                                   'making it an ideal companion for relaxing '
                                   'mornings or lively conversations. -- '
                                   "Ingredients: ['Espresso', 'Steamed Milk', "
                                   "'Milk Foam'] -- Price: 4.5 -- Rating: 4.7"},
              'score': 0.606236577,
              'values': []},
             {'id': 'Latte ',
              'metadata': {'text': 'Latte : Smooth and creamy, our latte '
                                   'combines rich espresso with velvety '
               